# Ablation Scenario Cost Strips

Strip plot showing relative cost distributions across all tuning runs (hazard_mult=0):
- rows: journey direction
- columns: forcing scenario (baseline, no_currents, no_waves, no_winds)
- color: speed

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

import pandas as pd
import geopandas as gpd
import numpy as np
from matplotlib import pyplot as plt
import cartopy
import cmocean
import xarray as xr
import seaborn as sns

from load_tuning_results import (
    load_results,
    get_forcing_df,
    get_seed_routes_gdf,
    filter_suspicious_routes,
    add_derived_features,
)

import warnings

warnings.filterwarnings("ignore")

In [ ]:
# parameters

# results dataframe from
gpq_file = "../results/results_prelim.geoparquet"

In [ ]:
gdf = gpd.read_parquet(gpq_file)
gdf = add_derived_features(gdf)
gdf = filter_suspicious_routes(gdf)
gdf

In [ ]:
# Filter to hazard_penalty_multiplier=0 only
_gdf = gdf[gdf.hyper_hazard_penalty_multiplier == 0].copy()

_gdf = _gdf.assign(journey_time_start=_gdf.journey_time_start.apply(lambda s: s[:10]))
print(f"Routes (hazard_mult=0): {len(_gdf)}")

In [ ]:
sns.set_context("paper", font_scale=1.5)

In [ ]:
g = sns.catplot(
    data=_gdf,
    x="elite_cost_relative",
    y="journey_time_start",
    row="journey_name",
    hue="journey_speed_knots",
    col="forcing_scenario_name",
    kind="strip",
    palette="Dark2",
    height=4,
    aspect=0.6,
    alpha=0.9,
    margin_titles=True,
)
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.legend.set_title("speed (kn)")
for lh in g.legend.legend_handles:
    lh.set_markersize(10)
sns.move_legend(g, "center right", bbox_to_anchor=(0.98, 0.5))

g.fig.savefig("../figures/023_ablation_scenario_cost_strips.pdf", dpi=200)
g.fig.savefig("../figures/023_ablation_scenario_cost_strips.png", dpi=200)